# Held-out validation — gold-labeled train / validation / test split

**Companion to `validation_prep.ipynb` and `validation_scoring.ipynb`.** Builds a single
gold-labeled dataset from the held-out double-coded sample and splits it into train (60%),
validation (30%) and test (10%) for training and evaluating a supervised classifier (per
construct and/or sentiment) using the human coding as ground truth, instead of — or alongside —
the dictionary.

**Gold label rule.** For each of the 8 binary constructs, the gold label is the majority vote
of three independent sources: the frozen dictionary, coder A, and coder B. Because there are
exactly three binary votes, a majority always exists (2–1 or 3–0) — no ties. For sentiment
(3-class), the gold label is the majority vote of the rating-derived `sentiment_class`, coder A's
independent judgement, and coder B's; a 3-way tie (all three different) falls back to the
rating-derived class and is flagged in `sentiment_n_agree = 1` for you to review by hand if you'd
rather adjudicate those cases manually.

**Split design.** `GroupShuffleSplit` cannot jointly hit target proportions across several
stratification dimensions at once — the same limitation flagged for the held-out sampler itself.
Instead, whole **providers** (`place_id`) are assigned to a single split — this prevents leakage
(the same establishment's reviews never appear in two splits) — via a greedy allocation that
processes each `country × mass_market` stratum in provider-size-descending order and, for every
provider, assigns it to whichever split is furthest below its target review count. Because this
provider-level allocation is size-aware and stratum-aware, the achieved split proportions track
the full held-out sample closely even on dimensions that were *not* used to stratify the provider
assignment (sentiment class, translation, review length) — the same effect the proportional
sampler in `validation_prep.ipynb` relies on.

**Run this after `validation_scoring.ipynb`** (or independently, as long as you have the same
inputs): the filled coder workbooks and `answer_key_PRIVATE.xlsx`. Nothing here touches the raw
corpus again.

**Naming.** `mass_market` is kept here as a column name for compatibility with `answer_key_PRIVATE.xlsx` and `tssc_analysis.ipynb`, but it records which retrieval query returned the provider, not a verified organizational type. It is referred to as the **query stratum** in the validation materials, and the grouped split below stratifies on it in that sense only — see section 6 of `validation_prep.ipynb`.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

CODER_A_PATH = Path("coder_A_blind.xlsx")     # filled in by coder A
CODER_B_PATH = Path("coder_B_blind.xlsx")     # filled in by coder B
KEY_PATH     = Path("answer_key_PRIVATE.xlsx")

OUT_DIR = Path("split_output")
OUT_DIR.mkdir(exist_ok=True)

FRACTIONS = {"train": 0.6, "validation": 0.3, "test": 0.1}
SEED = 20260817

CONSTRUCTS = ["disruption", "intermediation", "coordination", "delay",
              "digital", "price", "labour", "environmental"]
META_COLS = ["review_id", "place_id", "provider_pseudo", "country", "city",
             "mass_market", "translated", "n_words", "length_bucket"]

assert abs(sum(FRACTIONS.values()) - 1.0) < 1e-9, "FRACTIONS must sum to 1.0"
print("Reading from:", Path.cwd())

In [ ]:
key_A = pd.read_excel(KEY_PATH, sheet_name="coder_A_key")
key_B = pd.read_excel(KEY_PATH, sheet_name="coder_B_key")
coder_A = pd.read_excel(CODER_A_PATH, sheet_name="Coding")
coder_B = pd.read_excel(CODER_B_PATH, sheet_name="Coding")

missing_meta = [c for c in META_COLS if c not in key_A.columns]
if missing_meta:
    raise KeyError(
        f"answer_key_PRIVATE.xlsx is missing {missing_meta}. This notebook needs the enriched "
        "answer key produced by the current version of validation_prep.ipynb (includes country, "
        "city, mass_market, translated, n_words, length_bucket). Re-run validation_prep.ipynb to "
        "regenerate it."
    )

mA = key_A.merge(coder_A, on="validation_id", suffixes=("_key", "_coder"))
mB = key_B.merge(coder_B, on="validation_id", suffixes=("_key", "_coder"))

if set(mA.review_id) != set(mB.review_id):
    raise ValueError("Coder A and coder B workbooks do not cover the same set of reviews — "
                      "check that both files came from the same validation_prep.ipynb run.")

print(f"Coder A: {len(mA)} rows | Coder B: {len(mB)} rows | same review set: "
      f"{set(mA.review_id) == set(mB.review_id)}")

In [ ]:
# ---- build the gold-labeled base table (one row per review) -----------------------------------

base = mA[META_COLS + ["review_text"] + [f"{c}_key" for c in CONSTRUCTS] + ["sentiment_class"]].rename(
    columns={f"{c}_key": f"{c}_dict" for c in CONSTRUCTS}
)
base = base.merge(
    mA[["review_id"] + [f"{c}_coder" for c in CONSTRUCTS] + ["sentiment"]].rename(
        columns={**{f"{c}_coder": f"{c}_A" for c in CONSTRUCTS}, "sentiment": "sentiment_A"}),
    on="review_id",
)
base = base.merge(
    mB[["review_id"] + [f"{c}_coder" for c in CONSTRUCTS] + ["sentiment"]].rename(
        columns={**{f"{c}_coder": f"{c}_B" for c in CONSTRUCTS}, "sentiment": "sentiment_B"}),
    on="review_id",
)

# gold binary construct labels: majority vote of dictionary / coder A / coder B (never a tie)
for c in CONSTRUCTS:
    votes = base[[f"{c}_dict", f"{c}_A", f"{c}_B"]].astype(int)
    ones = votes.sum(axis=1)
    base[f"{c}_gold"] = (ones >= 2).astype(int)
    base[f"{c}_n_agree"] = np.maximum(ones, 3 - ones)  # 3 = unanimous, 2 = majority only

# gold sentiment: majority vote of rating-derived class / coder A / coder B; 3-way tie -> rating class
base["sentiment_A_norm"] = base["sentiment_A"].astype(str).str.strip().str.lower()
base["sentiment_B_norm"] = base["sentiment_B"].astype(str).str.strip().str.lower()
base["sentiment_class"] = base["sentiment_class"].astype(str).str.strip().str.lower()

def _sentiment_vote(row):
    votes = [row["sentiment_class"], row["sentiment_A_norm"], row["sentiment_B_norm"]]
    vals, counts = np.unique(votes, return_counts=True)
    top = counts.max()
    if top >= 2:
        return vals[counts.argmax()], int(top)
    return row["sentiment_class"], 1  # unresolved 3-way tie: default to rating-derived class

_sent = base.apply(_sentiment_vote, axis=1)
base["sentiment_gold"] = _sent.apply(lambda t: t[0])
base["sentiment_n_agree"] = _sent.apply(lambda t: t[1])

print(f"Gold-labeled base table: {len(base):,} reviews, {base.place_id.nunique():,} providers\n")
print("Construct agreement (votes out of 3 agreeing with the majority):")
for c in CONSTRUCTS:
    print(f"  {c:16s}", base[f"{c}_n_agree"].value_counts().sort_index(ascending=False).to_dict())
print("\nSentiment agreement:", base["sentiment_n_agree"].value_counts().sort_index(ascending=False).to_dict())
n_sentiment_ties = int((base["sentiment_n_agree"] == 1).sum())
if n_sentiment_ties:
    print(f"\n{n_sentiment_ties} review(s) had a 3-way sentiment disagreement (all three sources "
          "differ) and were defaulted to the rating-derived class. Consider adjudicating these "
          "by hand — see split_output/sentiment_ties_for_review.csv.")
    base[base.sentiment_n_agree == 1][
        ["review_id", "sentiment_class", "sentiment_A_norm", "sentiment_B_norm"]
    ].to_csv(OUT_DIR / "sentiment_ties_for_review.csv", index=False)

In [ ]:
# ---- grouped proportional train/validation/test split (no provider straddles a split) --------

def grouped_proportional_split(df, group_col, provider_strata_cols, fractions, seed):
    splits = list(fractions.keys())
    rng = np.random.default_rng(seed)

    prov = df.groupby(group_col).agg(
        n=("review_id", "size"),
        **{c: (c, "first") for c in provider_strata_cols},
    ).reset_index()
    prov["_stratum"] = prov[provider_strata_cols].astype(str).agg("|".join, axis=1)

    assigned = {}
    for _, grp in prov.groupby("_stratum"):
        stratum_total = grp["n"].sum()
        stratum_target = {s: stratum_total * fractions[s] for s in splits}
        stratum_achieved = {s: 0 for s in splits}
        grp = grp.sample(frac=1.0, random_state=rng.integers(0, 2**32 - 1)).sort_values("n", ascending=False)
        for _, row in grp.iterrows():
            deficits = {s: stratum_target[s] - stratum_achieved[s] for s in splits}
            best = max(deficits, key=deficits.get)
            assigned[row[group_col]] = best
            stratum_achieved[best] += row["n"]

    out = df.copy()
    out["split"] = out[group_col].map(assigned)
    return out

base = grouped_proportional_split(base, "place_id", ["country", "mass_market"], FRACTIONS, seed=SEED)

print("Achieved split sizes:")
print(base["split"].value_counts())
print("\nAchieved split shares (target: " +
      ", ".join(f"{k}={v:.0%}" for k, v in FRACTIONS.items()) + "):")
print((base["split"].value_counts(normalize=True) * 100).round(1))

prov_by_split = base.groupby("split")["place_id"].apply(set)
leaks = 0
splits = list(FRACTIONS.keys())
for i, a in enumerate(splits):
    for b in splits[i + 1:]:
        overlap = prov_by_split[a] & prov_by_split[b]
        leaks += len(overlap)
        if overlap:
            print(f"WARNING: {len(overlap)} provider(s) appear in both {a} and {b}")
print("\nProvider leakage across splits: 0 (each place_id lives in exactly one split)" if leaks == 0
      else f"\nProvider leakage across splits: {leaks} providers — investigate before training.")

In [ ]:
# ---- diversity check: each split vs. the full held-out sample ---------------------------------

print("--- Diversity check: split composition vs. full held-out sample ---")
for col in ["country", "mass_market", "sentiment_class", "translated", "length_bucket"]:
    full_p = base[col].value_counts(normalize=True).round(3)
    tab = {"full": full_p}
    for s in FRACTIONS:
        tab[s] = base.loc[base.split == s, col].value_counts(normalize=True).round(3)
    print(f"\n{col}")
    display(pd.DataFrame(tab).fillna(0))

In [ ]:
# ---- export gold-labeled train/validation/test tables + summary -------------------------------

EXPORT_COLS = (
    ["review_id", "place_id", "provider_pseudo", "country", "city", "mass_market",
     "translated", "n_words", "length_bucket", "review_text"]
    + [f"{c}_gold" for c in CONSTRUCTS] + [f"{c}_n_agree" for c in CONSTRUCTS]
    + ["sentiment_gold", "sentiment_n_agree"]
)

for split_name in FRACTIONS:
    subset = base.loc[base.split == split_name, EXPORT_COLS].reset_index(drop=True)
    subset.to_excel(OUT_DIR / f"{split_name}.xlsx", index=False)
    print(f"Wrote {OUT_DIR / f'{split_name}.xlsx'} ({len(subset):,} reviews, "
          f"{base.loc[base.split == split_name, 'place_id'].nunique():,} providers)")

summary = {
    "n_total": int(len(base)),
    "n_providers": int(base.place_id.nunique()),
    "target_fractions": FRACTIONS,
    "achieved_fractions": (base["split"].value_counts(normalize=True)).round(4).to_dict(),
    "split_sizes": base["split"].value_counts().to_dict(),
    "construct_gold_base_rate": {c: float(base[f"{c}_gold"].mean()) for c in CONSTRUCTS},
    "construct_unanimous_share": {c: float((base[f"{c}_n_agree"] == 3).mean()) for c in CONSTRUCTS},
    "sentiment_unanimous_share": float((base["sentiment_n_agree"] == 3).mean()),
    "sentiment_gold_distribution": base["sentiment_gold"].value_counts(normalize=True).round(4).to_dict(),
}
with open(OUT_DIR / "split_summary.json", "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

base[["review_id", "place_id", "split"]].to_csv(OUT_DIR / "split_assignment.csv", index=False)

print(json.dumps(summary, indent=2, ensure_ascii=False))
print(f"\nAll files written to {OUT_DIR.resolve()}")